In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_until_next_meeting AS (
WITH deduped_id AS (
    SELECT
        id AS activity_id,
        lead_id,
        status,
        starts_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        type = 'Meeting'
        AND status = 'upcoming'
        AND DATE(starts_at) > CURRENT_DATE
),
earliest_upcoming_per_lead AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY starts_at ASC) AS lead_id_rn
    FROM
        deduped_id
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(DATE(starts_at), CURRENT_DATE) AS days_until_next_meeting
FROM
    earliest_upcoming_per_lead
WHERE
    lead_id_rn = 1
)